# For colab environment

In [1]:
# !pip install nltk transformers==4.35.0 torch==2.6.0 torchvision==0.21.0 datasets accelerate==0.24.0 huggingface==0.0.1 datasets==2.14.7

In [1]:
import torch 
print(torch.cuda.is_available())
print(torch.__version__)

True
2.6.0+cu124


In [3]:
# !git clone https://github.com/BernardMoy/NLP-PCL-Classification.git

In [4]:
# %cd NLP-PCL-Classification/

In [2]:
!nvidia-smi

Mon Mar  2 15:12:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|


|   0  NVIDIA TITAN Xp                Off |   00000000:02:00.0  On |                  N/A |
| 23%   40C    P5             29W /  250W |     414MiB /  12288MiB |     12%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+
                                                                                         
+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI        PID   Type   Process name                              GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|    0   N/A  N/A     39465      G   /usr/bin/kalendarac                          

# Load train and validation data set

In [3]:
import pandas as pd 
import numpy as np 
from matplotlib import pyplot as plt
from collections import Counter

df = pd.read_csv('data/dontpatronizeme_pcl.tsv', sep='\t')

# Remove rows with NA labels 
df = df.dropna() 

# Add a bool_labels column for binary classification
df["bool_labels"] = df["label"] > 1   # is PCL if >1

# train val split 
train_labels = pd.read_csv('data/train_semeval_parids-labels.csv')["par_id"]
val_labels = pd.read_csv('data/dev_semeval_parids-labels.csv')["par_id"]
df_train = df[df["par_id"].isin(train_labels)].reset_index() 
df_val = df[df["par_id"].isin(val_labels)].reset_index() 


# Text Cleaning

In [4]:
# Remove special characters
SPECIAL_CHARACTERS = ['&amp;', '&lt;', '&gt;', '<h>', '\n', '\t']
for char in SPECIAL_CHARACTERS:
    df_train["text"] = df_train["text"].str.replace(char, "")
    df_val["text"] = df_val["text"].str.replace(char, "")


print(df_train["text"].iloc[55])


# Replace numbers with 0
df_train["text"] = df_train["text"].str.replace(r"\d+", "0", regex=True)
df_val["text"] = df_val["text"].str.replace(r"\d+", "0", regex=True)

print(df_train["text"].iloc[3])

People who are homeless , those who were once homeless , those working with the homeless and concerned New Zealanders are being asked to share their experiences and solutions to this growing issue with the Cross-Party Homelessness Inquiry . More
Council customers only signs would be displayed . Two of the spaces would be reserved for disabled persons and there would be five P0 spaces and eight P0 ones .


# Oversample the minority class
For each keyword category, inflate the number of positive examples to a certain percentage

In [5]:
POSITIVE_PERCENTAGE = 25


# Find all the unique keywords in the training dataset
keywords = pd.unique(df_train["keyword"])


# Extract the sub-dataset for each keyword
for keyword in keywords:
    subdata = df_train[df_train["keyword"] == keyword]
    rows = subdata.shape[0]


    # Find the number of positive entires x
    subdata_positive = subdata[subdata["bool_labels"] == True]
    positive_rows = subdata_positive.shape[0]


    # Calculate the number of additional samples needed to make the positive class reach the desired percentage
    # (p+x)/(r+x) = POS PERCENTAGE
    n_samples = round((100*positive_rows-POSITIVE_PERCENTAGE*rows)/(POSITIVE_PERCENTAGE-100)*1.0)


    # Sample with replacement from the sub dataset and add new rows
    sampled = subdata_positive.sample(n_samples, replace=True).reset_index(drop=True)
   
    # concat with the main df
    df_train = pd.concat([df_train, sampled], ignore_index=True)


df_train["bool_labels"].value_counts()


bool_labels
False    7581
True     2527
Name: count, dtype: int64

# Add contextual information to the text tokens

In [6]:
def add_info(df): 
    # Append the keyword and country code to the text, and separate them with additional separator tokens
    # Remove dashes in the keyword to match the format in the texts 
    return df["keyword"].str.replace('-', " ") + "<SEP>" + df["country_code"] + "<SEP>" + df["text"]

# Tokenization

In [7]:
from transformers import BertTokenizer, BertForSequenceClassification, AutoConfig, Trainer, TrainingArguments

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased") 

# define special tokens for separating text 
special_tokens = {"additional_special_tokens": ["<SEP>"]}
tokenizer.add_special_tokens(special_tokens) 

# Create text with contextual information 
def tokenize(df): 
    text_with_context = add_info(df)

    encoding = tokenizer(
        text_with_context.tolist(), 
        padding="max_length",   # Add padding to shorter sentences 
        max_length=256,
        truncation = True, 
        return_attention_mask = True 
    )

    return encoding

/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


# Convert to pyTorch dataset

In [8]:
import torch 
from torch.utils.data import DataLoader, TensorDataset
from datasets import Dataset

def to_dataset(df): 
    # Obtain tokens (input_ids, attention_mask) from the dataset 
    encoding = tokenize(df) 

    # Return huggingface dataset 
    return Dataset.from_dict({
        "input_ids": encoding["input_ids"], 
        "attention_mask": encoding["attention_mask"], 
        "label": df["bool_labels"].values 
    })

In [9]:
train_dataset = to_dataset(df_train)
val_dataset = to_dataset(df_val) 

# set to torch format 
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Training 

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=-1)

    # Calculate metrics 
    accuracy = accuracy_score(labels, predictions) 
    precision = precision_score(labels, predictions) 
    recall = recall_score(labels, predictions) 
    f1 = f1_score(labels, predictions) 

    return {
        "accuracy": accuracy, 
        "precision": precision, 
        "recall": recall, 
        "f1": f1 
    }


In [11]:
# Load roberta sequence classification model 
config = AutoConfig.from_pretrained("bert-base-uncased", num_labels=2)  # Binary classification
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", config = config)
model.resize_token_embeddings(len(tokenizer)) 

# Core hyperparameters 
BATCH_SIZE = 32
N_EPOCHS = 5 

# Set up training arguments 
training_args = TrainingArguments(
    fp16=True, 
    num_train_epochs=N_EPOCHS, 
    learning_rate=2e-5, 
    weight_decay=0.01,
    warmup_steps=500, 
    save_strategy="epoch",  # low disk space 
    load_best_model_at_end=True, 
    metric_for_best_model='f1',
    logging_steps=50,
    output_dir="./checkpoints/bert_ensemble", 
    evaluation_strategy="epoch", 
    per_device_eval_batch_size=BATCH_SIZE, 
    per_device_train_batch_size=BATCH_SIZE, 
)

# Set up trainer 
trainer = Trainer(
    model = model, 
    args = training_args, 
    train_dataset=train_dataset, 
    eval_dataset=val_dataset, 
    compute_metrics=compute_metrics
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [12]:
trainer.train() 

  0%|          | 0/1580 [00:00<?, ?it/s]

{'loss': 0.5927, 'learning_rate': 1.9600000000000003e-06, 'epoch': 0.16}
{'loss': 0.5364, 'learning_rate': 3.96e-06, 'epoch': 0.32}
{'loss': 0.5243, 'learning_rate': 5.92e-06, 'epoch': 0.47}
{'loss': 0.465, 'learning_rate': 7.92e-06, 'epoch': 0.63}
{'loss': 0.4106, 'learning_rate': 9.920000000000002e-06, 'epoch': 0.79}
{'loss': 0.4116, 'learning_rate': 1.1920000000000001e-05, 'epoch': 0.95}


  0%|          | 0/66 [00:00<?, ?it/s]

{'eval_loss': 0.25944650173187256, 'eval_accuracy': 0.8905876731963689, 'eval_precision': 0.4418604651162791, 'eval_recall': 0.5728643216080402, 'eval_f1': 0.4989059080962801, 'eval_runtime': 17.3427, 'eval_samples_per_second': 120.685, 'eval_steps_per_second': 3.806, 'epoch': 1.0}
{'loss': 0.3132, 'learning_rate': 1.392e-05, 'epoch': 1.11}
{'loss': 0.2952, 'learning_rate': 1.5920000000000003e-05, 'epoch': 1.27}
{'loss': 0.244, 'learning_rate': 1.792e-05, 'epoch': 1.42}
{'loss': 0.267, 'learning_rate': 1.9920000000000002e-05, 'epoch': 1.58}
{'loss': 0.2433, 'learning_rate': 1.9111111111111113e-05, 'epoch': 1.74}
{'loss': 0.2316, 'learning_rate': 1.8185185185185186e-05, 'epoch': 1.9}


  0%|          | 0/66 [00:00<?, ?it/s]

...

  0%|          | 0/66 [00:00<?, ?it/s]

{'eval_loss': 0.4168805480003357, 'eval_accuracy': 0.9182990922121357, 'eval_precision': 0.5769230769230769, 'eval_recall': 0.5276381909547738, 'eval_f1': 0.5511811023622047, 'eval_runtime': 17.2488, 'eval_samples_per_second': 121.342, 'eval_steps_per_second': 3.826, 'epoch': 4.0}
{'loss': 0.0159, 'learning_rate': 5.2222222222222226e-06, 'epoch': 4.11}
{'loss': 0.0061, 'learning_rate': 4.296296296296296e-06, 'epoch': 4.27}
{'loss': 0.0064, 'learning_rate': 3.3703703703703705e-06, 'epoch': 4.43}
{'loss': 0.0082, 'learning_rate': 2.4444444444444447e-06, 'epoch': 4.59}
{'loss': 0.0077, 'learning_rate': 1.5185185185185186e-06, 'epoch': 4.75}
{'loss': 0.0142, 'learning_rate': 5.925925925925927e-07, 'epoch': 4.91}


  0%|          | 0/66 [00:00<?, ?it/s]

{'eval_loss': 0.43116119503974915, 'eval_accuracy': 0.915910176779742, 'eval_precision': 0.5705521472392638, 'eval_recall': 0.46733668341708545, 'eval_f1': 0.5138121546961326, 'eval_runtime': 17.2512, 'eval_samples_per_second': 121.325, 'eval_steps_per_second': 3.826, 'epoch': 5.0}
{'train_runtime': 1312.3005, 'train_samples_per_second': 38.513, 'train_steps_per_second': 1.204, 'train_loss': 0.17383225454559809, 'epoch': 5.0}


TrainOutput(global_step=1580, training_loss=0.17383225454559809, metrics={'train_runtime': 1312.3005, 'train_samples_per_second': 38.513, 'train_steps_per_second': 1.204, 'train_loss': 0.17383225454559809, 'epoch': 5.0})

In [13]:
trainer.evaluate()

  0%|          | 0/66 [00:00<?, ?it/s]

{'eval_loss': 0.4168805480003357,
 'eval_accuracy': 0.9182990922121357,
 'eval_precision': 0.5769230769230769,
 'eval_recall': 0.5276381909547738,
 'eval_f1': 0.5511811023622047,
 'eval_runtime': 16.8627,
 'eval_samples_per_second': 124.12,
 'eval_steps_per_second': 3.914,
 'epoch': 5.0}

# Save model

In [15]:
trainer.save_model('models/model_bert_ensemble')

# Write Results

In [ ]:
import os 
# must pass abs path here 
bert_ensemble_model = BertForSequenceClassification.from_pretrained(os.path.abspath("/vol/bitbucket/bm1325/NLP-PCL-Classification/models/model_bert_ensemble"), local_files_only = True)
trainer = Trainer(model = bert_ensemble_model)

model_pred = trainer.predict(val_dataset)

# extract the class with higher logit value 
y_pred = model_pred.predictions.argmax(axis=1) 
y_true = df_val["bool_labels"]

with open("evaluation/bert_ensemble.txt", "w") as f: 
    f.write('\n'.join(str(x) for x in y_pred)) 


  0%|          | 0/262 [00:00<?, ?it/s]